# Gold Price Prediction using Linear and Ridge Regression
**Author:** Mohammad Gharebaghi  
**Project:** Time-Series Financial Modeling of SPDR Gold Shares (GLD)  

---

### Project Overview
This notebook implements an end-to-end regression workflow to predict **one-step-ahead closing prices** for the SPDR Gold Shares (`GLD`) ETF using daily market features (`Open`, `High`, `Low`, `Close`, `Volume`).

We benchmark:
1. **Ordinary Least Squares (OLS) Linear Regression** (Unpenalized baseline)
2. **Ridge Regression** ($L_2$ Tikhonov Regularization)

### Core Pipeline Steps
1. **Data Acquisition & Verification:** Ingestion of 7 years of daily GLD trading data (2018–2024).
2. **Target Formulation:** Shift target $y_t = \text{Close}_{t+1}$.
3. **Temporal Forward-Chaining Split:** Strict chronological 80/20 train/test partition without shuffling to prevent look-ahead data leakage.
4. **Feature Standardization:** Calibration of `StandardScaler` on the training partition to normalize disparate scales (price vs. volume).
5. **Model Estimation & Regularization Tuning:** Analytical formulation of OLS and Ridge, accompanied by cross-validation across penalty strength $\alpha$.
6. **Diagnostics & Evaluation:** Assessment via MAE, RMSE, $R^2$, MAPE, and residual error distributions.


## 1. Imports and Environment Setup


In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Matplotlib configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120
print("Environment successfully initialized.")


Environment successfully initialized.


## 2. Dataset Ingestion & Preprocessing


In [2]:
df = pd.read_csv('GLD.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

print(f"Total trading days: {len(df)}")
print(f"Date range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Missing values check:
{df.isnull().sum()}")
df.head()


Total trading days: 1761
Date range: 2018-01-02 to 2024-12-31
Missing values check:
Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64


## 3. Target Variable Construction
Forecasting requires using information available up to market close on day $t$ to anticipate the settlement price on day $t+1$:
$$y_t = \text{Close}_{t+1}$$
The final row lacks a subsequent price and is removed.


In [3]:
df['Target_Next_Close'] = df['Close'].shift(-1)
df_clean = df.dropna(subset=['Target_Next_Close']).copy().reset_index(drop=True)
print(f"Clean analytical dataset: {len(df_clean)} trading days.")
df_clean[['Date', 'Open', 'High', 'Low', 'Close', 'Target_Next_Close']].tail()


Clean analytical dataset: 1760 trading days.
           Date        Open        High         Low       Close  Target_Next_Close
1755 2024-12-23  237.580002  238.410004  236.850006  237.570007         241.699997
1756 2024-12-24  241.490005  242.080002  240.850006  241.699997         243.660004
1757 2024-12-26  242.389999  244.410004  242.270004  243.660004         242.360001
1758 2024-12-27  241.199997  243.080002  241.050003  242.360001         241.520004
1759 2024-12-30  241.080002  242.089996  240.669998  241.520004         241.020004


## 4. Chronological Train-Test Split (No Shuffling)
Financial data must **never be shuffled**. Shuffling introduces look-ahead bias (data leakage), where future price distributions inform past predictions.

We split chronologically at 80%:
- **Training Set (80%):** 1,408 days (January 2018 – August 2023)
- **Testing Set (20%):** 352 days (August 2023 – December 2024)


In [4]:
feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
target_col = 'Target_Next_Close'

split_idx = int(len(df_clean) * 0.8)
train_df = df_clean.iloc[:split_idx].copy().reset_index(drop=True)
test_df = df_clean.iloc[split_idx:].copy().reset_index(drop=True)

print(f"Training partition: {len(train_df)} samples ({train_df['Date'].min().date()} to {train_df['Date'].max().date()})")
print(f"Testing partition : {len(test_df)} samples ({test_df['Date'].min().date()} to {test_df['Date'].max().date()})")


Training partition: 1408 samples (2018-01-02 to 2023-08-07)
Testing partition : 352 samples (2023-08-08 to 2024-12-30)


## 5. Feature Standardization
Because Ridge regression applies an $L_2$ penalty ($\alpha \sum \beta_j^2$), unscaled features allow high-magnitude attributes (`Volume` $\sim 10^7$) to dominate, while lower-magnitude features (`Prices` $\sim 10^2$) are excessively penalized.

Standardization $z = (x - \mu) / \sigma$ is fitted **strictly on the training partition**.


In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feature_cols])
y_train = train_df[target_col].values

X_test = scaler.transform(test_df[feature_cols])
y_test = test_df[target_col].values

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")


X_train shape: (1408, 5), X_test shape: (352, 5)


## 6. Model Estimation: OLS vs. Ridge Regression
- **OLS:** $\mathbf{w}_{\text{OLS}} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$
- **Ridge:** $\mathbf{w}_{\text{Ridge}} = (\mathbf{X}^T \mathbf{X} + \alpha \mathbf{I})^{-1} \mathbf{X}^T \mathbf{y}$

Ridge introduces $\alpha \mathbf{I}$ to stabilize matrix inversion under extreme collinearity ($r > 0.999$).


In [6]:
# Ordinary Least Squares
lr = LinearRegression()
lr.fit(X_train, y_train)

# Ridge Regression (alpha = 1.0)
ridge_default = Ridge(alpha=1.0, random_state=42)
ridge_default.fit(X_train, y_train)

# Cross-Validated Ridge
alphas = np.logspace(-3, 3, 50)
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train, y_train)
best_alpha = ridge_cv.alpha_

ridge_tuned = Ridge(alpha=best_alpha, random_state=42)
ridge_tuned.fit(X_train, y_train)

print(f"Optimal Alpha discovered via 5-Fold CV: {best_alpha:.4f}")


Optimal Alpha discovered via 5-Fold CV: 0.0010


## 7. Performance Evaluation on Out-of-Sample Test Set


In [7]:
lr_pred = lr.predict(X_test)
rg_def_pred = ridge_default.predict(X_test)
rg_opt_pred = ridge_tuned.predict(X_test)

def evaluate(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': root_mean_squared_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
        'MAPE (%)': np.mean(np.abs((y_true - y_pred) / y_true)) * 100.0
    }

results = {
    'Linear Regression (OLS)': evaluate(y_test, lr_pred),
    'Ridge Regression (alpha=1.0)': evaluate(y_test, rg_def_pred),
    f'Ridge Regression (tuned alpha={best_alpha:.3f})': evaluate(y_test, rg_opt_pred)
}

res_df = pd.DataFrame(results).T
print("================ TEST SET PERFORMANCE ================")
print(res_df.to_string(formatters={'MAE': '{:.4f}'.format, 'RMSE': '{:.4f}'.format, 'R2': '{:.4f}'.format, 'MAPE (%)': '{:.4f}%'.format}))
print("======================================================")


================ TEST SET PERFORMANCE ================
                                              MAE    RMSE      R2  MAPE (%)
Linear Regression (OLS)                    1.4745  1.9759  0.9938   0.6902%
Ridge Regression (alpha=1.0)               1.4812  1.9699  0.9939   0.6953%
Ridge Regression (tuned alpha=0.001)       1.4745  1.9758  0.9938   0.6902%


### 7.1 Sample Daily Predictions


In [8]:
sample_mask = (test_df['Date'] >= '2024-10-01') & (test_df['Date'] <= '2024-10-08')
idx = test_df[sample_mask].index

sample_df = pd.DataFrame({
    'Date': test_df.loc[idx, 'Date'].dt.strftime('%Y-%m-%d'),
    'Actual Close': y_test[idx],
    'Predicted OLS': lr_pred[idx],
    'Predicted Ridge': rg_def_pred[idx]
})
print("Sample Daily Predictions (October 2024):")
print(sample_df.to_string(index=False, formatters={'Actual Close': '{:.2f}'.format, 'Predicted OLS': '{:.2f}'.format, 'Predicted Ridge': '{:.2f}'.format}))


Sample Daily Predictions (October 2024):
      Date  Actual Close  Predicted OLS  Predicted Ridge
2024-10-01        245.66         245.48           245.70
2024-10-02        245.49         245.58           245.28
2024-10-03        245.00         245.40           245.09
2024-10-04        244.17         244.80           245.06
2024-10-07        242.37         244.11           244.18
2024-10-08        241.05         242.14           242.20


## 8. Diagnostic Visualizations


In [9]:
# Actual vs Predicted Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
plt.subplots_adjust(hspace=0.35)

ax1.plot(test_df['Date'], y_test, label='Actual Next-Day Close', color='#1f77b4', linewidth=1.8)
ax1.plot(test_df['Date'], rg_def_pred, label='Ridge Regression', linestyle='--', color='#2ca02c', linewidth=1.4)
ax1.plot(test_df['Date'], lr_pred, label='OLS Linear Regression', linestyle=':', color='#ff7f0e', linewidth=1.2, alpha=0.8)
ax1.set_title('Out-of-Sample GLD Closing Price Tracking (2023–2024)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Price (USD)')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc='upper left')

# Zoom window
zoom_n = 60
ax2.plot(test_df['Date'].iloc[-zoom_n:], y_test[-zoom_n:], marker='o', markersize=3, label='Actual Next-Day Close', color='#1f77b4')
ax2.plot(test_df['Date'].iloc[-zoom_n:], rg_def_pred[-zoom_n:], marker='x', markersize=3, linestyle='--', label='Ridge Regression', color='#2ca02c')
ax2.plot(test_df['Date'].iloc[-zoom_n:], lr_pred[-zoom_n:], marker='.', markersize=3, linestyle=':', label='OLS', color='#ff7f0e')
ax2.set_title(f'Detailed Trajectory (Final {zoom_n} Trading Sessions)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Price (USD)')
ax2.set_xlabel('Date')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc='upper left')
plt.show()


In [10]:
# Residual Diagnostics
lr_res = y_test - lr_pred
rg_res = y_test - rg_def_pred

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plt.subplots_adjust(hspace=0.3, wspace=0.25)

axes[0, 0].plot(test_df['Date'], lr_res, color='#ff7f0e', alpha=0.7)
axes[0, 0].axhline(0, color='black', linestyle='--')
axes[0, 0].set_title('OLS Residuals Over Time', fontweight='bold')
axes[0, 0].set_ylabel('Error ($)')
axes[0, 0].grid(True, linestyle='--', alpha=0.5)

axes[0, 1].plot(test_df['Date'], rg_res, color='#2ca02c', alpha=0.7)
axes[0, 1].axhline(0, color='black', linestyle='--')
axes[0, 1].set_title('Ridge Residuals Over Time', fontweight='bold')
axes[0, 1].set_ylabel('Error ($)')
axes[0, 1].grid(True, linestyle='--', alpha=0.5)

axes[1, 0].hist(lr_res, bins=30, density=True, alpha=0.6, color='#ff7f0e', edgecolor='black', label=f'OLS (Std={np.std(lr_res):.2f})')
axes[1, 0].hist(rg_res, bins=30, density=True, alpha=0.4, color='#2ca02c', edgecolor='black', label=f'Ridge (Std={np.std(rg_res):.2f})')
axes[1, 0].set_title('Residual Density Distribution', fontweight='bold')
axes[1, 0].set_xlabel('Error ($)')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()
axes[1, 0].grid(True, linestyle='--', alpha=0.5)

axes[1, 1].scatter(y_test, lr_pred, alpha=0.4, color='#ff7f0e', s=15, label='OLS')
axes[1, 1].scatter(y_test, rg_def_pred, alpha=0.4, color='#2ca02c', s=15, label='Ridge')
mn, mx = min(y_test.min(), lr_pred.min()), max(y_test.max(), lr_pred.max())
axes[1, 1].plot([mn, mx], [mn, mx], 'k--', label='1:1 Line')
axes[1, 1].set_title('Actual vs. Predicted Scatter', fontweight='bold')
axes[1, 1].set_xlabel('Actual Close ($)')
axes[1, 1].set_ylabel('Predicted Close ($)')
axes[1, 1].legend()
axes[1, 1].grid(True, linestyle='--', alpha=0.5)
plt.show()


In [11]:
# Feature Correlation & Ridge Regularization Path
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
corr = df_clean[feature_cols + ['Target_Next_Close']].corr()
im = ax1.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
fig.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
cols = feature_cols + ['Target']
ax1.set_xticks(range(len(cols)))
ax1.set_yticks(range(len(cols)))
ax1.set_xticklabels(cols, rotation=45, ha='right')
ax1.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax1.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black" if abs(corr.iloc[i, j]) < 0.7 else "white", fontsize=8)
ax1.set_title("Market Feature Correlation Matrix", fontweight="bold")

# Bias-variance path
alphas_test = np.logspace(-3, 4, 100)
rmses = [root_mean_squared_error(y_test, Ridge(alpha=a).fit(X_train, y_train).predict(X_test)) for a in alphas_test]
ax2.semilogx(alphas_test, rmses, color='#d62728', linewidth=2.0, label='Test RMSE')
ax2.axvline(x=alphas_test[np.argmin(rmses)], color='black', linestyle='--', label=f'Min RMSE Alpha ({alphas_test[np.argmin(rmses)]:.3f})')
ax2.set_title('Bias-Variance Curve: Test RMSE vs. Alpha', fontweight='bold')
ax2.set_xlabel('Regularization Penalty Alpha (log scale)')
ax2.set_ylabel('Test RMSE (USD)')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend()
plt.show()


## 9. Key Findings & Engineering Insights
1. **Multicollinearity Impact:** Pairwise correlation among `Open`, `High`, `Low`, and `Close` exceeds $0.999$, making unpenalized OLS coefficient estimates unstable. Ridge Regression introduces the positive diagonal perturbation $\alpha \mathbf{I}$, shrinking erratic weights and reducing out-of-sample RMSE ($1.9699$ vs. $1.9759$).
2. **Standardization Necessity:** Without standardization, share volume (order of $10^7$) dominates the $L_2$ regularizer, rendering price weights unregularized.
3. **Temporal Partitioning:** Chronological splitting ensures zero look-ahead bias, reflecting realistic out-of-sample forecasting performance.
4. **Autoregressive Level Persistence:** Financial price series closely follow geometric random walk dynamics ($P_{t+1} = P_t + \epsilon_t$), which naturally leads to high $R^2$ scores since today's price anchors the nominal level.
